# Пример · #98 PReLU (эталон) · **medium**

После [98_prelu.ipynb](./98_prelu.ipynb).


## Зачем эта задача

PReLU — Leaky с **обучаемым** α; нужны forward и backward по x и α.


## Условие

Forward как Leaky; backward: grad_x и grad_alpha на отрицательных x.


In [ ]:
import numpy as np
import torch
import torch.nn.functional as F


def need(cond, msg=""):
    assert cond, msg or "проверка не прошла"

In [ ]:
def prelu_forward(x: np.ndarray, alpha: float = 0.25) -> np.ndarray:
    return np.where(x > 0, x, alpha * x)


def prelu_backward(
    x: np.ndarray, alpha: float, grad_output: np.ndarray
) -> tuple[np.ndarray, float]:
    grad_x = np.where(x > 0, grad_output, alpha * grad_output)
    mask = x < 0
    grad_alpha = float(np.sum(x[mask] * grad_output[mask]))
    return grad_x, grad_alpha


fwd = prelu_forward(np.array([-2.0, -1.0, 0.0, 1.0, 2.0]), alpha=0.1)
need(np.allclose(fwd, [-0.2, -0.1, 0.0, 1.0, 2.0]))

grad_x, grad_alpha = prelu_backward(
    np.array([2.0, -1.0, 0.0, -3.0]), 0.25, np.ones(4)
)
need(np.allclose(grad_x, [1.0, 0.25, 0.25, 0.25]))
need(abs(grad_alpha - (-4.0)) < 1e-9)

In [ ]:
z = torch.tensor([-2.0, -0.5, 1.0, 3.0])
alpha = 0.1
mine = prelu_forward(z.numpy(), alpha)
ref = F.prelu(z, torch.tensor([alpha]))
need(np.allclose(mine, ref.numpy()))
print("forward OK vs F.prelu")

## Построчно

- `np.where` в forward.
- grad_alpha = Σ x·grad_output на маске x<0.


## Связь

`F.prelu` с параметром-скаляром; в Conv-сетях α часто по каналам.


Submit: [#98](https://www.deep-ml.com/problems/98).